# Phase 0 — threshold sensitivity grid

Referee 2 charges that "PAV + genetic support in 2–5 therapeutic areas" was chosen to maximise the
odds ratio. Before any held-out validation, the cheapest and most informative thing to know is what
the *neighbouring* definitions give. Two possibilities with opposite consequences for the response:

- **A broad plateau around 2–5.** Then the exact window is not load-bearing, most of the charge
  dissolves without any hold-out at all, and the response can say so with the full surface attached.
- **A sharp spike at 2–5.** Then the published number is a local maximum of noise, we must know it
  before the referee computes it himself, and the framing has to change to a bias-corrected estimate.

This notebook computes the full surface and reports it. No selection, no thresholds chosen here.

Inputs: `ti_pairs_chembl_master-r1.parquet` from `01_build_pair_tables.ipynb`. Statistics come from
`or10_stats.py`, whose `or_rs` reproduces the published enrichment implementation exactly.

In [1]:
import numpy as np
import pandas as pd

from or10_stats import or_rs, support_mask, window_label

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

path_to_intermediate_data_folder = "../../../data/intermediate_files/"
master = pd.read_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet")
print(master.shape)
print("approved (phase 4) pairs:", int(master["approved"].sum()))
master.head()

(37377, 10)
approved (phase 4) pairs: 4564


,targetId,diseaseId,maxClinicalPhase,score_all,score_pav,uniqueTherapeuticAreas,uniqueDiseases,approvedSymbol,in_gps,approved
0,ENSG00000007314,EFO_0000555,2.0,NaN,NaN,NaN,NaN,None,False,0
1,ENSG00000007314,EFO_0004699,3.0,NaN,NaN,NaN,NaN,None,False,0
2,ENSG00000007314,EFO_0801084,2.0,NaN,NaN,NaN,NaN,None,False,0
3,ENSG00000010310,EFO_0003884,2.0,NaN,NaN,9.0,15.0,GIPR,True,0
4,ENSG00000012504,MONDO_0019052,4.0,NaN,NaN,NaN,NaN,None,False,1


## Self-check

The module functions must still return the published values on this table, otherwise nothing below
is comparable to the manuscript.

In [2]:
baseline = or_rs(support_mask(master), master["approved"])
published = or_rs(support_mask(master, pav=True, ta_min=2, ta_max=5), master["approved"])

assert np.isclose(baseline["odds_ratio"], 3.618578, rtol=1e-5), baseline["odds_ratio"]
assert np.isclose(published["odds_ratio"], 10.288962, rtol=1e-5), published["odds_ratio"]
assert np.isclose(published["relative_success"], 4.843708, rtol=1e-5), published["relative_success"]
assert published["yes_evid-high_clinphase"] == 51  # the manuscript says 52; see notebook 01

BASELINE_OR = baseline["odds_ratio"]
PUBLISHED_OR = published["odds_ratio"]
PUBLISHED_RS = published["relative_success"]
print(f"all-GWAS baseline OR = {BASELINE_OR:.3f}, RS = {baseline['relative_success']:.3f}")
print(
    f"published strict OR  = {PUBLISHED_OR:.3f} [{published['ci_low']:.2f}, {published['ci_high']:.2f}], "
    f"RS = {PUBLISHED_RS:.3f}"
)

all-GWAS baseline OR = 3.619, RS = 2.765
published strict OR  = 10.289 [6.71, 15.78], RS = 4.844


## Cross-check against the published regression table

`df_for_enrichment_regression.csv` is the pair-level table the published non-linearity analysis
used. It is an independent construction of the same object (it carries `max_vep` aggregated over the
supporting credible sets rather than a separately propagated PAV score), so agreeing with it on
every pair is a real check on the PAV column, not a tautology.

In [3]:
published_pairs = pd.read_csv(path_to_intermediate_data_folder + "df_for_enrichment_regression.csv")
print(published_pairs.shape)

merged = master.merge(
    published_pairs[["targetId", "diseaseId", "geneticSupport", "max_vep", "outcome", "uniqueTherapeuticAreas"]],
    on=["targetId", "diseaseId"],
    how="inner",
    suffixes=("", "_pub"),
)
assert len(merged) == len(master) == len(published_pairs), (len(merged), len(master), len(published_pairs))

assert (merged["score_all"].notna() == (merged["geneticSupport"] == 1)).all(), "all-GWAS support disagrees"
assert (merged["score_pav"].notna() == ((merged["geneticSupport"] == 1) & (merged["max_vep"] == 1))).all(), (
    "PAV support disagrees with the published max_vep flag"
)
assert (merged["approved"] == merged["outcome"]).all(), "outcome disagrees"
assert (merged["uniqueTherapeuticAreas"].fillna(0) == merged["uniqueTherapeuticAreas_pub"]).all(), (
    "therapeutic-area counts disagree"
)
print("pair-level master agrees with df_for_enrichment_regression.csv on support, PAV, outcome and TA count")

(37377, 11)
pair-level master agrees with df_for_enrichment_regression.csv on support, PAV, outcome and TA count


## The grid

`evaluate` runs one definition; `grid_over` maps it across a list of definitions. Every row carries
the 2×2 counts, so nothing in the surface can be read without also seeing how thin it is.

In [4]:
def evaluate(df, pav, ta_min, ta_max, extra=None):
    """One definition: OR, RS, CIs and the 2x2 counts."""
    mask = support_mask(df, pav=pav, ta_min=ta_min, ta_max=ta_max)
    row = {
        "pav": pav,
        "ta_min": ta_min,
        "ta_max": ta_max,
        "window": window_label(ta_min, ta_max),
        **or_rs(mask, df["approved"]),
    }
    row["definition"] = ("PAV" if pav else "any") + " + TA " + row["window"]
    if extra:
        row.update(extra)
    return row


def grid_over(df, definitions):
    """Evaluate a list of (pav, ta_min, ta_max) definitions."""
    rows = [evaluate(df, pav, lo, hi) for pav, lo, hi in definitions]
    out = pd.DataFrame(rows)
    out["or_vs_baseline"] = out["odds_ratio"] / BASELINE_OR
    out["ci_excludes_baseline"] = out["ci_low"] > BASELINE_OR
    return out


REPORT_COLUMNS = [
    "definition",
    "odds_ratio",
    "ci_low",
    "ci_high",
    "relative_success",
    "ci_rs_low",
    "ci_rs_high",
    "p_value",
    "n_support",
    "yes_evid-high_clinphase",
    "or_vs_baseline",
    "ci_excludes_baseline",
]

### The windows named in the analysis brief

`all` = no window at all (the published all-GWAS definition when PAV is off).

In [5]:
named_windows = [(1, 4), (1, 5), (2, 4), (2, 5), (2, 6), (3, 6), (2, None), (None, None)]
named = grid_over(master, [(pav, lo, hi) for pav in (True, False) for lo, hi in named_windows])
named_report = named[REPORT_COLUMNS].round(4)
named_report

,definition,odds_ratio,ci_low,ci_high,relative_success,ci_rs_low,ci_rs_high,p_value,n_support,yes_evid-high_clinphase,or_vs_baseline,ci_excludes_baseline
0,PAV + TA 1-4,8.7392,5.5408,13.7841,4.5085,3.6622,5.5503,0.0,75,41,2.4151,True
1,PAV + TA 1-5,8.9924,5.9809,13.5203,4.5711,3.8039,5.4930,0.0,94,52,2.4851,True
2,PAV + TA 2-4,10.3527,6.3808,16.7970,4.8511,3.9689,5.9295,0.0,68,40,2.8610,True
3,PAV + TA 2-5,10.2890,6.7079,15.7819,4.8437,4.0513,5.7912,0.0,87,51,2.8434,True
4,PAV + TA 2-6,9.4670,6.3546,14.1038,4.6776,3.9275,5.5709,0.0,99,56,2.6162,True
5,PAV + TA 3-6,8.5561,5.5783,13.1236,4.4669,3.6658,5.4431,0.0,85,46,2.3645,True
6,PAV + TA >=2,6.2315,4.5323,8.5676,3.8196,3.2129,4.5407,0.0,154,71,1.7221,True
7,PAV + TA all,5.8934,4.3129,8.0533,3.7051,3.1136,4.4089,0.0,161,72,1.6287,True
8,any + TA 1-4,4.1416,3.3565,5.1103,3.0103,2.6280,3.4483,0.0,386,139,1.1445,False
9,any + TA 1-5,4.0169,3.3069,4.8794,2.9541,2.6008,3.3554,0.0,457,161,1.1101,False


In [6]:
print("PAV definitions ranked by OR:")
print(
    named[named["pav"]]
    .sort_values("odds_ratio", ascending=False)[
        ["window", "odds_ratio", "ci_low", "ci_high", "n_support", "yes_evid-high_clinphase"]
    ]
    .round(3)
    .to_string(index=False)
)

PAV definitions ranked by OR:
window  odds_ratio  ci_low  ci_high  n_support  yes_evid-high_clinphase
   2-4      10.353   6.381   16.797         68                       40
   2-5      10.289   6.708   15.782         87                       51
   2-6       9.467   6.355   14.104         99                       56
   1-5       8.992   5.981   13.520         94                       52
   1-4       8.739   5.541   13.784         75                       41
   3-6       8.556   5.578   13.124         85                       46
   >=2       6.231   4.532    8.568        154                       71
   all       5.893   4.313    8.053        161                       72


### Full surface over (lower bound, upper bound)

Every window with `1 <= ta_min <= 8`, `ta_max` from `ta_min` up to 12 plus unbounded, crossed with
PAV on and off. This is deliberately larger than any window anyone would report — the point is to
show the whole landscape the published choice sits in, including how many definitions would have
beaten it.

In [7]:
ta_values = list(range(1, 9))
upper_values = list(range(1, 13)) + [None]

full_definitions = [
    (pav, lo, hi) for pav in (True, False) for lo in ta_values for hi in upper_values if hi is None or hi >= lo
]
full = grid_over(master, full_definitions)
print("definitions evaluated:", len(full))
print("with at least 10 approved supported pairs:", int((full["yes_evid-high_clinphase"] >= 10).sum()))

definitions evaluated: 152
with at least 10 approved supported pairs: 138


In [8]:
# OR surface for the PAV definitions, rows = lower bound, columns = upper bound
pav_surface = full[full["pav"]].copy()
pav_surface["ta_max_label"] = pav_surface["ta_max"].apply(lambda v: "inf" if v is None or pd.isna(v) else str(int(v)))
or_surface = pav_surface.pivot(index="ta_min", columns="ta_max_label", values="odds_ratio")
or_surface = or_surface[[c for c in [str(i) for i in range(1, 13)] + ["inf"] if c in or_surface.columns]]
or_surface.round(2)

ta_max_label,1,2,3,4,5,6,7,8,9,10,11,12,inf
ta_min,,,,,,,,,,,,,
1,1.2,7.93,7.62,8.74,8.99,8.46,7.65,7.28,6.95,6.86,6.88,5.89,5.89
2,NaN,18.01,10.82,10.35,10.29,9.47,8.39,7.91,7.52,7.39,7.39,6.23,6.23
3,NaN,NaN,7.20,9.04,9.29,8.56,7.56,7.12,6.75,6.66,6.68,5.61,5.61
4,NaN,NaN,NaN,9.93,9.95,8.88,7.61,7.09,6.66,6.56,6.58,5.42,5.42
5,NaN,NaN,NaN,NaN,9.91,7.69,6.01,5.48,5.03,5.01,5.13,4.07,4.07
6,NaN,NaN,NaN,NaN,NaN,5.14,4.05,3.77,3.46,3.60,3.84,3.07,3.07
7,NaN,NaN,NaN,NaN,NaN,NaN,3.20,3.08,2.80,3.09,3.44,2.70,2.70
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.88,2.40,3.00,3.60,2.55,2.55


In [9]:
# how thin each cell is: approved pairs carrying the estimate
n_surface = pav_surface.pivot(index="ta_min", columns="ta_max_label", values="yes_evid-high_clinphase")
n_surface = n_surface[[c for c in [str(i) for i in range(1, 13)] + ["inf"] if c in n_surface.columns]]
n_surface

ta_max_label,1,2,3,4,5,6,7,8,9,10,11,12,inf
ta_min,,,,,,,,,,,,,
1,1.0,11.0,19.0,41.0,52.0,57.0,61.0,63.0,64.0,66.0,68.0,72.0,72.0
2,NaN,10.0,18.0,40.0,51.0,56.0,60.0,62.0,63.0,65.0,67.0,71.0,71.0
3,NaN,NaN,8.0,30.0,41.0,46.0,50.0,52.0,53.0,55.0,57.0,61.0,61.0
4,NaN,NaN,NaN,22.0,33.0,38.0,42.0,44.0,45.0,47.0,49.0,53.0,53.0
5,NaN,NaN,NaN,NaN,11.0,16.0,20.0,22.0,23.0,25.0,27.0,31.0,31.0
6,NaN,NaN,NaN,NaN,NaN,5.0,9.0,11.0,12.0,14.0,16.0,20.0,20.0
7,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6.0,7.0,9.0,11.0,15.0,15.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,5.0,7.0,11.0,11.0


### Is 2–5 a plateau or a spike?

Three questions, answered on the PAV surface restricted to windows with at least 10 approved
supported pairs (below that the odds ratio is not interpretable and the maximum is guaranteed to be
noise):

1. Where does the published window rank?
2. How many windows sit within the published 95% CI — i.e. are statistically indistinguishable?
3. What is the spread of the odds ratio across the neighbourhood of 2–5?

In [10]:
MIN_APPROVED = 10

usable = full[(full["pav"]) & (full["yes_evid-high_clinphase"] >= MIN_APPROVED)].copy()
usable = usable.sort_values("odds_ratio", ascending=False).reset_index(drop=True)
usable["rank"] = usable.index + 1

pub_row = usable[(usable["ta_min"] == 2) & (usable["ta_max"] == 5)].iloc[0]
print(f"PAV windows with >= {MIN_APPROVED} approved supported pairs: {len(usable)}")
print(f"published window 2-5: rank {int(pub_row['rank'])} of {len(usable)}, OR = {pub_row['odds_ratio']:.3f}")
print(
    f"max OR in the grid: {usable.iloc[0]['definition']} OR = {usable.iloc[0]['odds_ratio']:.3f} "
    f"({int(usable.iloc[0]['yes_evid-high_clinphase'])} approved pairs)"
)
print(f"min OR in the grid: {usable.iloc[-1]['definition']} OR = {usable.iloc[-1]['odds_ratio']:.3f}")
print()
within_ci = usable[(usable["odds_ratio"] >= pub_row["ci_low"]) & (usable["odds_ratio"] <= pub_row["ci_high"])]
print(
    f"windows whose OR falls inside the published 95% CI [{pub_row['ci_low']:.2f}, {pub_row['ci_high']:.2f}]: "
    f"{len(within_ci)} of {len(usable)}"
)
print(f"windows with OR above the published point estimate: {int((usable['odds_ratio'] > PUBLISHED_OR).sum())}")
print(
    f"windows whose CI excludes the all-GWAS baseline {BASELINE_OR:.2f}: "
    f"{int(usable['ci_excludes_baseline'].sum())} of {len(usable)}"
)
print()
print("top 15 by OR:")
print(
    usable.head(15)[["rank", "definition", "odds_ratio", "ci_low", "ci_high", "n_support", "yes_evid-high_clinphase"]]
    .round(3)
    .to_string(index=False)
)

PAV windows with >= 10 approved supported pairs: 64
published window 2-5: rank 4 of 64, OR = 10.289
max OR in the grid: PAV + TA 2 OR = 18.011 (10 approved pairs)
min OR in the grid: PAV + TA >=8 OR = 2.555

windows whose OR falls inside the published 95% CI [6.71, 15.78]: 32 of 64
windows with OR above the published point estimate: 3
windows whose CI excludes the all-GWAS baseline 3.62: 45 of 64

top 15 by OR:
 rank   definition  odds_ratio  ci_low  ci_high  n_support  yes_evid-high_clinphase
    1   PAV + TA 2      18.011   5.646   57.452         14                       10
    2 PAV + TA 2-3      10.823   5.210   22.484         30                       18
    3 PAV + TA 2-4      10.353   6.381   16.797         68                       40
    4 PAV + TA 2-5      10.289   6.708   15.782         87                       51
    5 PAV + TA 4-5       9.950   5.876   16.850         57                       33
    6   PAV + TA 4       9.929   5.211   18.919         38                       

In [11]:
# immediate neighbourhood of the published window: move either bound by one
neighbours = [(2, 5), (1, 5), (3, 5), (2, 4), (2, 6), (1, 4), (1, 6), (3, 4), (3, 6)]
nb = grid_over(master, [(True, lo, hi) for lo, hi in neighbours])
nb_report = nb[
    ["definition", "odds_ratio", "ci_low", "ci_high", "relative_success", "n_support", "yes_evid-high_clinphase"]
].round(3)
print(nb_report.to_string(index=False))
print()
print(f"OR range across the 3x3 neighbourhood: {nb['odds_ratio'].min():.2f} - {nb['odds_ratio'].max():.2f}")
print(
    f"relative spread: {100 * (nb['odds_ratio'].max() - nb['odds_ratio'].min()) / nb['odds_ratio'].median():.0f}% of the median"
)

  definition  odds_ratio  ci_low  ci_high  relative_success  n_support  yes_evid-high_clinphase
PAV + TA 2-5      10.289   6.708   15.782             4.844         87                       51
PAV + TA 1-5       8.992   5.981   13.520             4.571         94                       52
PAV + TA 3-5       9.286   5.842   14.759             4.632         73                       41
PAV + TA 2-4      10.353   6.381   16.797             4.851         68                       40
PAV + TA 2-6       9.467   6.355   14.104             4.678         99                       56
PAV + TA 1-4       8.739   5.541   13.784             4.508         75                       41
PAV + TA 1-6       8.456   5.765   12.404             4.447        106                       57
PAV + TA 3-4       9.040   5.280   15.476             4.573         54                       30
PAV + TA 3-6       8.556   5.578   13.124             4.467         85                       46

OR range across the 3x3 neighbourhood: 

### Odds ratio by exact therapeutic-area count

The window was motivated by a non-linear pleiotropy effect: enrichment rising then falling with the
number of therapeutic areas. This is that curve without any windowing — each row is one exact TA
count, PAV required. If the published window brackets a genuine peak, the peak should be visible
here; if the curve is flat with noisy ends, the window is doing the work.

In [12]:
profile_rows = []
for k in range(1, 13):
    row = evaluate(master, True, k, k)
    row["ta_count"] = k
    profile_rows.append(row)
row = evaluate(master, True, 13, None)
row["ta_count"] = 13  # 13+
profile_rows.append(row)

profile = pd.DataFrame(profile_rows)
profile["ta_count_label"] = profile["ta_count"].astype(str).where(profile["ta_count"] < 13, "13+")
print(
    profile[
        [
            "ta_count_label",
            "odds_ratio",
            "ci_low",
            "ci_high",
            "relative_success",
            "n_support",
            "yes_evid-high_clinphase",
        ]
    ]
    .round(3)
    .to_string(index=False)
)

ta_count_label  odds_ratio  ci_low  ci_high  relative_success  n_support  yes_evid-high_clinphase
             1       1.198   0.144    9.956             1.170          7                        1
             2      18.011   5.646   57.452             5.860         14                       10
             3       7.200   2.701   19.195             4.100         16                        8
             4       9.929   5.211   18.919             4.759         38                       22
             5       9.907   3.983   24.643             4.750         19                       11
             6       5.140   1.631   16.202             3.415         12                        5
             7       3.197   0.984   10.387             2.521         13                        4
             8       2.877   0.558   14.832             2.340          7                        2
             9       1.798   0.201   16.087             1.638          5                        1
            10      

### Secondary grid — gPS (distinct diseases) windows

The published text also reports a gPS split (`gPS <= 5` OR 4.8 versus `gPS >= 10` OR 3.0). The same
sensitivity question applies to that threshold, so the surface is computed for gPS windows too. This
is secondary: the claim under attack is stated in therapeutic areas.

In [13]:
gps_rows = []
for pav in (True, False):
    for lo, hi in [(1, 3), (1, 5), (1, 10), (2, 5), (2, 10), (2, 20), (3, 10), (5, None), (10, None), (None, None)]:
        mask = support_mask(master, pav=pav, gps_min=lo, gps_max=hi)
        gps_rows.append(
            {
                "pav": pav,
                "gps_min": lo,
                "gps_max": hi,
                "definition": ("PAV" if pav else "any") + " + gPS " + window_label(lo, hi),
                **or_rs(mask, master["approved"]),
            }
        )
gps = pd.DataFrame(gps_rows)
gps["or_vs_baseline"] = gps["odds_ratio"] / BASELINE_OR
print(
    gps[["definition", "odds_ratio", "ci_low", "ci_high", "relative_success", "n_support", "yes_evid-high_clinphase"]]
    .round(3)
    .to_string(index=False)
)

    definition  odds_ratio  ci_low  ci_high  relative_success  n_support  yes_evid-high_clinphase
 PAV + gPS 1-3       5.892   2.440   14.226             3.691         20                        9
 PAV + gPS 1-5       9.022   4.672   17.423             4.565         36                       20
PAV + gPS 1-10       7.427   4.799   11.494             4.174         81                       41
 PAV + gPS 2-5      11.427   5.543   23.555             5.036         31                       19
PAV + gPS 2-10       8.050   5.126   12.642             4.340         76                       40
PAV + gPS 2-20       7.107   4.768   10.595             4.085         97                       48
PAV + gPS 3-10       7.676   4.766   12.363             4.240         68                       35
 PAV + gPS >=5       5.520   3.907    7.801             3.568        132                       57
PAV + gPS >=10       4.661   3.097    7.015             3.227         97                       38
 PAV + gPS all      

## Export

`or10_phase0_grid_full-r1.csv` is the full surface, the input to the Phase 0 heatmap when figures
are made. Figures are deliberately not produced at this stage.

In [14]:
named.to_csv(path_to_intermediate_data_folder + "or10_phase0_grid_named-r1.csv", index=False)
full.to_csv(path_to_intermediate_data_folder + "or10_phase0_grid_full-r1.csv", index=False)
profile.to_csv(path_to_intermediate_data_folder + "or10_phase0_ta_profile-r1.csv", index=False)
gps.to_csv(path_to_intermediate_data_folder + "or10_phase0_grid_gps-r1.csv", index=False)
print("exported 4 tables")

exported 4 tables
